# GPAP — Cross-GPU Benchmark

Measure **how fast GPAP propagates assumption sets on *this* GPU vs a sequential CPU baseline.**

For each bundled CNF the notebook runs the full GPAP pipeline over all (capped) ± variable
**pairs** and compares wall-clock against a sequential CPU [PySAT] loop over the *same* sets:

> **speedup = CPU_wall / GPU_wall = GPU sets/s ÷ CPU sets/s**

Triplets and a random-`L` assumption-size sweep are reported as supporting context. Results are
saved per-GPU to `benchmarks/colab_results/<gpu>.json`, so running this notebook on different
Colab GPUs lets you rank them (last cell).

**Before running:** `Runtime ▸ Change runtime type ▸ GPU` (T4/L4/A100…). The batch size `K`
auto-sizes to whatever VRAM the GPU has — no manual tuning.

## 1. Setup — clone repo + install deps
Torch ships preinstalled on Colab with the right CUDA build, so we only add `python-sat` + `tqdm`.

In [ ]:
import os, subprocess, sys
from getpass import getpass

REPO_OWNER = "yogevshalmon"
REPO_NAME  = "gpap"
BRANCH     = "collab_enb"

# Preferred: store your PAT as a Colab Secret named GITHUB_TOKEN
# (left sidebar → key icon → 'Add new secret').
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    token = ""
if not token:
    print("No GITHUB_TOKEN secret found.")
    print("Create one at GitHub → Settings → Developer settings → Personal access tokens")
    print("(scope: Contents read — or 'repo' for full access)")
    token = getpass("GitHub PAT (leave blank if repo is public): ")

auth     = f"{token}@" if token else ""
repo_url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"

if os.path.basename(os.getcwd()) != REPO_NAME:
    if not os.path.isdir(REPO_NAME):
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, REPO_NAME],
            check=True,
        )
    os.chdir(REPO_NAME)
subprocess.run(["git", "pull", "--ff-only", repo_url, BRANCH], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-sat", "tqdm"], check=True)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("repo:", os.getcwd(), "| branch:", BRANCH)


## 2. Detect the GPU
Confirms which accelerator this run used (recorded in the result file).

In [ ]:
import torch
from scripts.colab_benchmark import run_all, save_results, compare_saved, gpu_info

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
info = gpu_info(device)
for k, v in info.items():
    print(f"{k:>16}: {v}")
if info.get("total_gb") is None:
    print("\n  No CUDA GPU detected — set Runtime > Change runtime type > GPU and re-run cell 1.")

## 3. Run the benchmark
Two measurements, both full end-to-end (generate → reset → apply → propagate → extract):

- **Pairs anchor** — exhaustive (±v1,±v2) sets, GPU vs sequential CPU; the headline sets/s + a conflict sanity check.
- **Random-L × soft-stop grid** — for each L∈{2,5,10,20}, random L-literal sets streamed in `target_chunks×K` batches, run with no cutoff and with several `soft_stop_frac` values (min_rounds=5). Reports `sets/s` and `successful_sets/s` (finished = ¬timed_out).

`quick=True` finishes in a few minutes (subset of instances, L∈{2,10}, fracs {0.5,0.1}, target_chunks=6).
Set `quick=False` for the full run (all instances, L∈{2,5,10,20}, fracs {0.9,0.7,0.5,0.3,0.1}, target_chunks=32) — minutes to ~1 h depending on the GPU. `K` auto-sizes to all available VRAM.

In [ ]:
payload = run_all(quick=True)

## 4. Save this GPU's result
Writes `benchmarks/colab_results/<gpu-slug>.json` (one file per GPU; re-runs overwrite).

In [ ]:
path = save_results(payload)

# Optional: download a copy to your machine.
try:
    from google.colab import files
    files.download(path)
except Exception:
    pass

## 4b. Plots
Throughput vs formula size (by literals), speedup vs M, and the soft-stop tradeoff curve.

In [ ]:
from scripts.colab_plots import make_plots
from scripts.colab_benchmark import DEFAULT_OUT_DIR
from IPython.display import Image, display

for p in make_plots(payload, DEFAULT_OUT_DIR):
    display(Image(filename=p))

## 5. (Optional) Commit the result back to the repo
Persist this GPU's numbers in version control so they accumulate across GPUs. Needs a GitHub
**Personal Access Token** with `repo` write scope. **Never hard-code a token you intend to keep** —
paste it here for the session only, or use Colab Secrets.

Uncomment, fill in, and run:

In [ ]:
# from getpass import getpass
# if not token:  # reuse token from cell 1 if already set
#     token = getpass("GitHub PAT: ")
# GIT_NAME, GIT_EMAIL = "Your Name", "you@example.com"
#
# subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
# subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)
# subprocess.run(["git", "add", "benchmarks/colab_results/"], check=True)  # JSON + PNGs
# subprocess.run(["git", "commit", "-m", f"colab: {payload['gpu']['name']} benchmark result"], check=True)
# remote = f"https://{token}@github.com/yogevshalmon/gpap.git"
# subprocess.run(["git", "push", remote, f"HEAD:{BRANCH}"], check=True)
# print("pushed")


## 6. Compare GPUs
Ranks every result JSON currently in `benchmarks/colab_results/` by peak pairs throughput.
Pull the repo (cell 1 re-run) to fetch results committed from other GPUs first.

In [ ]:
_ = compare_saved()